In [9]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile, is_zipfile
import shutil

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

BASE_URL = "https://raw.githubusercontent.com/idanm-commiting/Sanitation-Project/main/"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def load_github_zip(filename, record_type):
    """Download and cache a ZIP, then read all CSV members without extracting."""
    archive_path = DATA_DIR / filename
    if not archive_path.exists() or not is_zipfile(archive_path):
        print(f"Downloading {filename} ...", flush=True)
        temporary_path = archive_path.with_suffix(".zip.part")
        with urlopen(BASE_URL + filename, timeout=120) as response:
            with temporary_path.open("wb") as destination:
                shutil.copyfileobj(response, destination)
        if not is_zipfile(temporary_path):
            raise ValueError(f"The download for {filename} is not a valid ZIP file.")
        temporary_path.replace(archive_path)

    frames = []
    with ZipFile(archive_path) as archive:
        csv_members = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv") and not name.startswith("__MACOSX/")
        ]
        if not csv_members:
            raise ValueError(f"No CSV files found in {filename}.")
        for member in csv_members:
            with archive.open(member) as csv_file:
                frame = pd.read_csv(csv_file, low_memory=False)
            frame.columns = frame.columns.str.strip()
            # Align shared fields before stacking the two kinds of records.
            frame = frame.rename(columns={
                "Created Date": "Creation Date",
                "Address": "Street Address",
                "Zip": "ZIP Code",
            })
            date_column = "Creation Date" if record_type == "311 complaint" else "Inspection Date"
            frame["Record Date"] = pd.to_datetime(
                frame[date_column], errors="coerce", format="mixed"
            )
            frame["Record Type"] = record_type
            frame["Source File"] = filename
            frames.append(frame)
    result = pd.concat(frames, ignore_index=True, sort=False)
    return result


# Assign each archive to its own DataFrame.
df_311 = load_github_zip("311_Sanitation_Combined_2010_2026.zip", "311 complaint")
df_food_part_1 = load_github_zip("Food_Inspections_part_1.zip", "Food inspection")
df_food_part_2a = load_github_zip("Food_Inspections_part_2a.zip", "Food inspection")
df_food_part_2b = load_github_zip("Food_Inspections_part_2b.zip", "Food inspection")
df_food_part_3 = load_github_zip("Food_Inspections_part_3.zip", "Food inspection")

# Stack all five DataFrames into one combined dataset.
df = pd.concat(
    [df_food_part_1, df_food_part_2a, df_food_part_2b, df_food_part_3],
    ignore_index=True,
    sort=False,
)



In [14]:
print(df.columns)
print(df_311.columns)



Index(['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type',
       'Risk', 'Street Address', 'City', 'State', 'ZIP Code',
       'Inspection Date', 'Inspection Type', 'Results', 'Violations',
       'Latitude', 'Longitude', 'Location', 'Year', 'Record Date',
       'Record Type', 'Source File'],
      dtype='object')
Index(['Request Number', 'Creation Date', 'Closed Date', 'Status',
       'Request Type', 'Street Address', 'ZIP Code', 'Ward', 'Police District',
       'Community Area', 'Latitude', 'Longitude', 'Year', 'Record Date',
       'Record Type', 'Source File'],
      dtype='object')


In [17]:
merged_df = pd.merge(
    df_311,
    df,
    on=['Street Address', 'Year'],
    how='inner'
)

merged_df.columns

Index(['Request Number', 'Creation Date', 'Closed Date', 'Status',
       'Request Type', 'Street Address', 'ZIP Code_x', 'Ward',
       'Police District', 'Community Area', 'Latitude_x', 'Longitude_x',
       'Year', 'Record Date_x', 'Record Type_x', 'Source File_x',
       'Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type',
       'Risk', 'City', 'State', 'ZIP Code_y', 'Inspection Date',
       'Inspection Type', 'Results', 'Violations', 'Latitude_y', 'Longitude_y',
       'Location', 'Record Date_y', 'Record Type_y', 'Source File_y'],
      dtype='object')